# Chapter 2 — From $k$-NN to Kernels

Companion notebook to Chapter 2 of *Kernels and Transformers for Tabular Data*.

Reproduces:
- Figure 2.1: $k$-NN decision boundary at $k \in \{1, 5, 25\}$.
- Figure 2.2: NW estimator with RBF kernel at varying bandwidth.
- Figure 2.3: Mercer eigendecomposition of an RBF Gram matrix.
- Numerical verification of representer theorem (Theorem 4).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from tabkernels.classical import KNNClassifier, KNNRegressor, NadarayaWatson, KernelRidgeRegression, rbf_kernel

rng = np.random.RandomState(42)
np.random.seed(42)
# Find the figures dir robustly regardless of cwd.
import os
_p = os.getcwd()
while _p and not os.path.isdir(os.path.join(_p, 'affinity', 'book')):
    _p = os.path.dirname(_p)
FIGURES_DIR = os.path.join(_p, 'affinity', 'book', 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

## Figure 2.1: $k$-NN decision boundary

In [ ]:
X, y = make_moons(n_samples=200, noise=0.20, random_state=42)
X = X.astype(np.float32); y = y.astype(np.int64)
x1, x2 = np.meshgrid(np.linspace(-1.5, 2.5, 200), np.linspace(-1.0, 1.5, 200))
grid = np.c_[x1.ravel(), x2.ravel()].astype(np.float32)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, k in zip(axes, [1, 5, 25]):
    m = KNNClassifier(k=k).fit(X, y)
    Z = m.predict(grid).reshape(x1.shape)
    ax.contourf(x1, x2, Z, alpha=0.3, levels=[-0.5, 0.5, 1.5], cmap='RdBu')
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='RdBu', s=20, edgecolors='k', linewidth=0.4)
    ax.set_title(f'$k = {k}$'); ax.set_xticks([]); ax.set_yticks([])
fig.suptitle('Figure 2.1: $k$-NN decision boundary on two moons')
plt.tight_layout(); plt.savefig(f'{FIGURES_DIR}/fig_02_01_knn_boundary.pdf', bbox_inches='tight')
plt.show()

## Figure 2.2: Nadaraya–Watson at varying bandwidth

In [ ]:
x_train = np.linspace(-3, 3, 80).astype(np.float32)[:, None]
y_train = (np.sin(x_train.ravel()) + 0.2 * np.random.randn(80)).astype(np.float32)
x_grid = np.linspace(-3, 3, 300).astype(np.float32)[:, None]

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, sigma in zip(axes, [0.1, 0.5, 2.0]):
    m = NadarayaWatson(bandwidth=sigma).fit(x_train, y_train)
    yhat = m.predict(x_grid)
    ax.scatter(x_train, y_train, s=10, alpha=0.5, label='train')
    ax.plot(x_grid, np.sin(x_grid.ravel()), 'k--', lw=1, label='truth')
    ax.plot(x_grid, yhat, 'r-', lw=2, label=f'NW ($\\sigma$={sigma})')
    ax.set_title(f'$\\sigma = {sigma}$'); ax.legend(loc='upper left', fontsize=8)
fig.suptitle('Figure 2.2: Nadaraya--Watson with RBF kernel; bandwidth controls bias-variance')
plt.tight_layout(); plt.savefig(f'{FIGURES_DIR}/fig_02_02_nw_bandwidth.pdf', bbox_inches='tight')
plt.show()

## Figure 2.3: Mercer eigendecomposition of RBF Gram matrix

In [ ]:
X = rng.randn(100, 2).astype(np.float32)
K = rbf_kernel(X, X, sigma=1.0)
eigvals, eigvecs = np.linalg.eigh(K)
eigvals = eigvals[::-1]; eigvecs = eigvecs[:, ::-1]  # descending

fig, axes = plt.subplots(1, 4, figsize=(15, 3.3))
axes[0].semilogy(eigvals[:30] + 1e-12, 'o-')
axes[0].set_title('Top 30 eigenvalues (log)'); axes[0].set_xlabel('index'); axes[0].grid(alpha=0.3)
for i, ax in enumerate(axes[1:]):
    sc = ax.scatter(X[:, 0], X[:, 1], c=eigvecs[:, i], cmap='RdBu', s=30)
    ax.set_title(f'$\\phi_{i+1}(x)$'); ax.set_xticks([]); ax.set_yticks([])
    plt.colorbar(sc, ax=ax, fraction=0.046)
fig.suptitle('Figure 2.3: Mercer decomposition of RBF Gram matrix on 100 random 2D points')
plt.tight_layout(); plt.savefig(f'{FIGURES_DIR}/fig_02_03_mercer.pdf', bbox_inches='tight')
plt.show()

## Numerical check: representer theorem (Theorem 4)

Solve KRR; verify the predictor at any point lies in the span of training-kernel slices.

In [ ]:
X_tr = rng.randn(40, 3).astype(np.float32)
y_tr = (X_tr[:, 0] + 0.5 * X_tr[:, 1] ** 2 + 0.1 * np.random.randn(40)).astype(np.float32)
krr = KernelRidgeRegression(bandwidth=1.0, ridge=0.01).fit(X_tr, y_tr)
X_q = rng.randn(8, 3).astype(np.float32)
yhat_direct = krr.predict(X_q)
K_q_tr = rbf_kernel(X_q, X_tr, sigma=1.0)
yhat_representer = K_q_tr @ krr.dual_coefs
diff = np.abs(yhat_direct - yhat_representer).max()
print(f'Max |direct - representer-form| = {diff:.2e}')
print('Representer theorem verified numerically: predictor is a finite kernel sum.')

## Bias–variance: the $k$-NN/NW progression

Sweep $k$ for $k$-NN and $\sigma$ for NW; show test MSE on a held-out split.

In [ ]:
X = rng.randn(200, 2).astype(np.float32)
y = (np.sin(X[:, 0]) + np.cos(X[:, 1]) + 0.2 * np.random.randn(200)).astype(np.float32)
X_tr, X_te = X[:150], X[150:]; y_tr, y_te = y[:150], y[150:]
ks = [1, 3, 5, 10, 25, 50]
sigmas = [0.05, 0.1, 0.3, 0.5, 1.0, 2.0]
knn_mse = [((KNNRegressor(k=k).fit(X_tr, y_tr).predict(X_te) - y_te) ** 2).mean() for k in ks]
nw_mse = [((NadarayaWatson(bandwidth=s).fit(X_tr, y_tr).predict(X_te) - y_te) ** 2).mean() for s in sigmas]
print('k-NN test MSE by k:', dict(zip(ks, [round(m, 3) for m in knn_mse])))
print('NW   test MSE by sigma:', dict(zip(sigmas, [round(m, 3) for m in nw_mse])))

**End of notebook.** Reproduces all 3 figures plus a numerical representer-theorem check and a bias–variance sweep.